导入依赖

In [1]:
import torch
import torchvision
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from models.MobileNetV3 import MobileNetV3

设置随机种子

In [2]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # 如果使用多GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed = 100
# 在训练代码开头调用
set_seed(seed) # 42, 2025, 1024, 512, 100

导入数据集

In [3]:
def preprocessing(tensor):
    if tensor.mean() > 0.5:
        tensor = 1 - tensor
    return tensor

In [4]:
def tensor_transform(x):
    if x.shape[1] == 1:
        return x.repeat_interleave(3, dim= 1)
    elif x.shape[1] == 4:
        return x[:, :3, :, :]
    elif x.shape[1] == 3:
        return x
    else:
        raise ValueError('Invalid input shape')

tensor_rgb_transform = transforms.Lambda(tensor_transform)

In [5]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Lambda(preprocessing),
    transforms.Normalize((0.1307, ), (0.3081, )),
    transforms.Grayscale(1)
])

In [6]:
mnist_db = datasets.MNIST(root= 'dataFolder/', train= True, transform= transform, download= True)
mnist_loader = DataLoader(mnist_db, batch_size= 128, shuffle= True)

In [7]:
our_db = datasets.ImageFolder(root= 'dataFolder/num', transform= transform)
our_loader = DataLoader(our_db, batch_size= 128, shuffle= True)

模型训练

In [8]:
def train_model(model, train_loader, in_ch):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr= 0.001)
    model.train()

    losses = []
    for epoch in range(10):  # 训练10个周期
        correct = 0
        total = 0
        for idx, (x, y) in enumerate(train_loader):
            x, y = x.to(device), y.to(device)
            if in_ch == 3: x = tensor_rgb_transform(x)
            out = model(x)
            loss = criterion(out, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += x.shape[0]
            correct += (torch.argmax(out, dim= -1) == y).sum().item()
            losses.append(loss.item())
            if idx % 1000 == 0:
                print('Epoch: {}, Loss: {}'.format(epoch, loss.item()))
                print('Accuracy: {:.2f}%'.format(correct * 100 / total))

    return model.state_dict(), losses

In [9]:
model_1C = MobileNetV3(version= 'small', n_classes= 10, in_ch= 1)
state_dict, losses = train_model(model_1C, mnist_loader, in_ch= 1)
torch.save(state_dict, f'save_models/mobV3/MobileNetV3_1C_MD_{seed}.pth')
print('MobileNetV3_1C saved!')

model_3C = MobileNetV3(version= 'small', n_classes= 10, in_ch= 3)
state_dict, losses = train_model(model_3C, mnist_loader, in_ch= 3)
torch.save(state_dict, f'save_models/mobV3/MobileNetV3_3C_MD_{seed}.pth')
print('MobileNetV3_1C saved!')

model_1C_expSE = MobileNetV3(version= 'small', n_classes= 10, in_ch= 1, use_se= False)
state_dict, losses = train_model(model_1C_expSE, mnist_loader, in_ch= 1)
torch.save(state_dict, f'save_models/mobV3/MobileNetV3_expSE_1C_MD_{seed}.pth')
print('MobileNetV3_expSE_1C saved!')

model_3C_expSE = MobileNetV3(version= 'small', n_classes= 10, in_ch= 3, use_se= False)
state_dict, losses = train_model(model_3C_expSE, mnist_loader, in_ch= 3)
torch.save(state_dict, f'save_models/mobV3/MobileNetV3_expSE_3C_MD_{seed}.pth')
print('MobileNetV3_expSE_3C saved!')

# model_1C = MobileNetV3(version= 'small', n_classes= 10, in_ch= 1)
# state_dict, losses = train_model(model_1C, our_loader, in_ch= 1)
# torch.save(state_dict, 'save_models/OD/MobileNetV3_1C_OD_42.pth')
# print('MobileNetV3_1C saved!')

# model_3C = MobileNetV3(version= 'small', n_classes= 10, in_ch= 3)
# state_dict, losses = train_model(model_3C, our_loader, in_ch= 3)
# torch.save(state_dict, 'save_models/OD/MobileNetV3_3C_OD_42.pth')
# print('MobileNetV3_1C saved!')

# model_1D = MobileNetV3(version= 'small', n_classes= 10, in_ch= 1)
# state_dict, losses = train_model(model_1D, mnist_loader, in_ch= 1)
# torch.save(state_dict, f'save_models/mobV3/MobileNetV3_se_1C_MD_{seed}.pth')
# print('MobileNetV3_1C saved!')

# model_3D = MobileNetV3(version= 'small', n_classes= 10, in_ch= 3)
# state_dict, losses = train_model(model_3D, mnist_loader, in_ch= 3)
# torch.save(state_dict, f'save_models/mobV3/MobileNetV3_se_3C_MD_{seed}.pth')
# print('MobileNetV3_1C saved!')

Epoch: 0, Loss: 2.2973034381866455
Accuracy: 11.72%
Epoch: 1, Loss: 0.0629047080874443
Accuracy: 97.66%
Epoch: 2, Loss: 0.08286873251199722
Accuracy: 96.88%
Epoch: 3, Loss: 0.06071909889578819
Accuracy: 97.66%
Epoch: 4, Loss: 0.024755191057920456
Accuracy: 98.44%
Epoch: 5, Loss: 0.003164163790643215
Accuracy: 100.00%
Epoch: 6, Loss: 0.05097329989075661
Accuracy: 99.22%
Epoch: 7, Loss: 0.007837869226932526
Accuracy: 99.22%
Epoch: 8, Loss: 0.0012327481526881456
Accuracy: 100.00%
Epoch: 9, Loss: 0.020416000857949257
Accuracy: 99.22%
MobileNetV3_1C saved!
Epoch: 0, Loss: 2.306819438934326
Accuracy: 9.38%
Epoch: 1, Loss: 0.006573827471584082
Accuracy: 100.00%
Epoch: 2, Loss: 0.04545624926686287
Accuracy: 97.66%
Epoch: 3, Loss: 0.0016310346545651555
Accuracy: 100.00%
Epoch: 4, Loss: 0.002194360364228487
Accuracy: 100.00%
Epoch: 5, Loss: 0.010189248248934746
Accuracy: 99.22%
Epoch: 6, Loss: 0.034675829112529755
Accuracy: 99.22%
Epoch: 7, Loss: 0.014529624953866005
Accuracy: 100.00%
Epoch: 8, 